# Aurora Business Lab — Autonomous Economic & RL Training Notebook (Kaggle / Google Colab)

Ce notebook reproductible entraîne et évalue les politiques de décision économique d'**Aurora Business Intelligence** :
- **Séparation chronologique stricte** : 60% Entraînement / 20% Validation / 20% Test jamais vu (Walk-Forward).
- **Fonction de récompense multi-critères de survie** : pénalise fortement la ruine, le drawdown maximal et toute stratégie de type Martingale.
- **Compatibilité** : Kaggle (GPU T4x2 / P100 / CPU), Google Colab et exécution locale.

In [ ]:
import json
import math
import os
import random
import time
from pathlib import Path

SEED = 2026
random.seed(SEED)
print('Aurora Business Lab Training Environment initialized (seed =', SEED, ')')

In [ ]:
# Vérification optionnelle du GPU (PyTorch sur Kaggle / Colab) avec repli CPU déterministe
try:
    import torch
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    torch.manual_seed(SEED)
except ImportError:
    DEVICE = 'cpu'
print('Compute device:', DEVICE)

In [ ]:
# Si le dépôt AuroraIA est présent, utilise directement application/business_intelligence.py
import sys
for cand in [Path.cwd() / 'application', Path.cwd().parent / 'application']:
    if (cand / 'business_intelligence.py').exists():
        sys.path.insert(0, str(cand))
        break

from business_intelligence import (
    EconomicMemoryStore,
    run_advanced_policy_training,
    run_13_step_council_deliberation,
)

store = EconomicMemoryStore(Path.cwd())
training_report = run_advanced_policy_training(
    store=store,
    capital_eur=10000.0,
    episodes=80,
    horizon_months=24,
    compute_target=f'notebook_{DEVICE}',
)
print('Best Strategy:', training_report['learned_best_strategy_name'])
print('Promotion Gate:', json.dumps(training_report['promotion_gate'], indent=2))

In [ ]:
# Export du checkpoint vérifié pour réimport dans AuroraIA
out_path = Path('aurora_policy_checkpoint.json')
out_path.write_text(json.dumps(training_report, indent=2, ensure_ascii=False), encoding='utf-8')
print('Saved checkpoint to', out_path.resolve())